# RSNA — package a fixed 2,000-image development subset for download

**Runs on Kaggle. Data acquisition only.** No preprocessing, no model, no metrics.

The competition data is already mounted at `/kaggle/input`, so nothing is downloaded
from the Kaggle API here — no `kaggle.json`, no rules-acceptance 403, no per-file HTTP.
This selects 2,000 DICOMs and writes them into split `.tar` parts you download from the
**Output** tab.

### Why tar parts rather than one archive

`/kaggle/working` holds about 20 GB. Two thousand DICOMs is roughly 11 GB, so copying
the files *and then* archiving them would not fit. The files are therefore streamed
straight from `/kaggle/input` into archives — read once, written once, 11 GB total.

They are split into ~1.5 GB parts because an 11 GB single-file browser download that
fails at 90% has to start over, while a part can be retried on its own. Each part gets
a SHA-256 so you can verify it on your PC rather than hoping.

No compression: DICOM pixel data is already compressed, so gzip would cost minutes and
save almost nothing.

### The sealed holdout

Selection uses `development` only. Section 2 stops unless the split is exactly
6,214 / 2,663 with zero overlap, and section 4 stops again if any selected patient
appears in the holdout list.

## 1. Configuration

In [130]:
TARGET_IMAGES = 2000
SEED          = 42
PART_GB       = 1.5        # per-part size; smaller means more, safer downloads

EXPECTED_DEV     = 6214
EXPECTED_HOLDOUT = 2663

OUT      = '/kaggle/working/rsna_subset_%d' % TARGET_IMAGES
META_DIR = OUT + '/metadata'
PART_DIR = OUT + '/parts'

META_CSV   = META_DIR + '/rsna_validation_%d_metadata.csv' % TARGET_IMAGES
SELECT_CSV = META_DIR + '/rsna_validation_%d_selection.csv' % TARGET_IMAGES
SUMMARY    = META_DIR + '/selection_summary.json'
MANIFEST   = META_DIR + '/parts_manifest.json'

# Section 5 measures the real total and sets this itself when the subset fits
# /kaggle/working comfortably. It only stays False, and asks you, when the size is
# close enough to the ~20 GB cap that packaging could fail partway.
CONFIRM_PACKAGE = False
WORKING_LIMIT_GB = 20.0
SAFE_GB = 16.0          # below this, proceed without asking

import os, sys, json, time, random, hashlib, tarfile, glob, shutil
from datetime import datetime, timezone
import pandas as pd

for d in (META_DIR, PART_DIR):
    os.makedirs(d, exist_ok=True)
print('output   :', OUT)
print('target   :', TARGET_IMAGES, 'images')
print('seed     :', SEED)
print('part size: %.1f GB' % PART_GB)

output   : /kaggle/working/rsna_subset_2000
target   : 2000 images
seed     : 42
part size: 1.5 GB


## 2. Locate the inputs, and the first hard gate

The competition directory and the split file are found by name, so the dataset slug does
not matter. `/kaggle/working` is searched for the split too, in case you rebuilt it in
this session.

In [131]:
def find_one(patterns, what):
    for pat in patterns:
        hits = sorted({os.path.realpath(p) for p in glob.glob(pat, recursive=True)})
        if hits:
            print('%-18s %s' % (what, hits[0]))
            if len(hits) > 1:
                print('%-18s (%d candidates; using the first)' % ('', len(hits)))
            return hits[0]
    raise SystemExit('%s not found. Searched:\n%s'
                     % (what, '\n'.join('  ' + p for p in patterns)))


_tc = find_one(['/kaggle/input/**/rsna-breast-cancer-detection/train.csv',
                '/kaggle/input/**/train.csv'], 'train.csv')
COMP_DIR = os.path.dirname(_tc)
TRAIN_CSV = _tc
IMG_ROOT = os.path.join(COMP_DIR, 'train_images')
if not os.path.isdir(IMG_ROOT):
    raise SystemExit('train_images/ not found under %s - is the competition attached?'
                     % COMP_DIR)
print('%-18s %s' % ('train_images', IMG_ROOT))

SPLIT_FILE = find_one(['/kaggle/working/**/rsna_dev_holdout_split.json',
                       '/kaggle/input/**/rsna_dev_holdout_split.json'], 'split file')

with open(SPLIT_FILE) as fh:
    SPLIT = json.load(fh)
DEV = {str(p) for p in SPLIT.get('development', [])}
HOLD = {str(p) for p in SPLIT.get('holdout', [])}

G = []
print('\ndevelopment patients : %6d  (expected %d)' % (len(DEV), EXPECTED_DEV))
print('holdout patients     : %6d  (expected %d)   SEALED'
      % (len(HOLD), EXPECTED_HOLDOUT))
if len(DEV) != EXPECTED_DEV:
    G.append('development is %d, expected %d' % (len(DEV), EXPECTED_DEV))
if len(HOLD) != EXPECTED_HOLDOUT:
    G.append('holdout is %d, expected %d' % (len(HOLD), EXPECTED_HOLDOUT))
_ov = DEV & HOLD
print('holdout overlap      : %6d %s' % (len(_ov), '[PASS]' if not _ov else ''))
if _ov:
    G.append('%d patients in BOTH partitions' % len(_ov))
if G:
    for g in G:
        print('  GATE FAILED:', g)
    raise SystemExit('split gates failed - nothing selected, nothing packaged')
print('\nsplit gates passed')

train.csv          /kaggle/input/competitions/rsna-breast-cancer-detection/train.csv
train_images       /kaggle/input/competitions/rsna-breast-cancer-detection/train_images
split file         /kaggle/working/rsna_dev_holdout_split.json

development patients :   6214  (expected 6214)
holdout patients     :   2663  (expected 2663)   SEALED
holdout overlap      :      0 [PASS]

split gates passed


## 3. Select

Every cancer-positive image in development, then negatives sampled with `SEED = 42`
until the total is exactly 2,000.

This is an **image-level** selection, as specified. Noting once, so it does not surprise
you later: two views of the same breast are correlated, so 2,000 images are fewer than
2,000 independent observations. That affects how a confidence interval should be
computed downstream, not this packaging step — and the metadata keeps `patient_id`,
`laterality` and `view` so a later notebook can group by breast.

In [132]:
df = pd.read_csv(TRAIN_CSV)
df['patient_id'] = df['patient_id'].astype(str)
df['image_id'] = df['image_id'].astype(str)
print('train.csv rows : %d' % len(df))
print('columns        : %s' % ', '.join(df.columns))

dev_df = df[df['patient_id'].isin(DEV)].copy()
print('\ndevelopment    : %d images, %d patients'
      % (len(dev_df), dev_df['patient_id'].nunique()))
if set(dev_df['patient_id']) & HOLD:
    raise SystemExit('HOLDOUT LEAK in the development frame')

pos = dev_df[dev_df['cancer'] == 1].copy()
neg = dev_df[dev_df['cancer'] == 0].copy()
print('cancer-positive: %d images' % len(pos))
print('cancer-negative: %d images' % len(neg))

need = TARGET_IMAGES - len(pos)
if len(pos) > TARGET_IMAGES:
    raise SystemExit(
        '%d positives exceed the %d target. Raise TARGET_IMAGES or decide explicitly '
        'which positives to drop - this notebook will not choose for you.'
        % (len(pos), TARGET_IMAGES))
if need > len(neg):
    raise SystemExit('need %d negatives, only %d available' % (need, len(neg)))

rng = random.Random(SEED)
neg_idx = sorted(rng.sample(list(neg.index), need))
neg_sel = neg.loc[neg_idx].copy()

pos['selection_type'] = 'all_positive'
neg_sel['selection_type'] = 'sampled_negative'
sel = pd.concat([pos, neg_sel], ignore_index=True)
sel['partition'] = 'development'
sel['dicom_path'] = ['train_images/%s/%s.dcm' % (p, i)
                     for p, i in zip(sel['patient_id'], sel['image_id'])]
sel['source_path'] = [os.path.join(IMG_ROOT, p, '%s.dcm' % i)
                      for p, i in zip(sel['patient_id'], sel['image_id'])]

print('\nselected malignant : %d' % int((sel['cancer'] == 1).sum()))
print('selected negative  : %d' % int((sel['cancer'] == 0).sum()))
print('total selected     : %d' % len(sel))

train.csv rows : 54706
columns        : site_id, patient_id, image_id, laterality, view, age, cancer, biopsy, invasive, BIRADS, implant, density, machine_id, difficult_negative_case

development    : 29184 images, 6214 patients
cancer-positive: 800 images
cancer-negative: 28384 images

selected malignant : 800
selected negative  : 1200
total selected     : 2000


## 4. Summary and the remaining gates

In [133]:
print('Development patients available : %d' % len(DEV))
print('Holdout patients               : %d' % len(HOLD))
print('Holdout overlap                : %d [PASS]' % len(DEV & HOLD))
print()
print('Cancer-positive images available : %d' % len(pos))
print('Cancer-negative images available : %d' % len(neg))
print()
print('Selected malignant images : %d' % int((sel['cancer'] == 1).sum()))
print('Selected negative images  : %d' % int((sel['cancer'] == 0).sum()))
print('Total selected images     : %d' % len(sel))
print()
print('Unique patients           : %d' % sel['patient_id'].nunique())
for col, label in (('laterality', 'Left/right distribution'),
                   ('view', 'CC/MLO distribution'),
                   ('site_id', 'Site distribution'),
                   ('machine_id', 'Machine distribution')):
    if col in sel.columns:
        print('\n%s:' % label)
        print(sel.groupby(col).size().to_string())

GG = []
if set(sel['patient_id']) & HOLD:
    GG.append('a selected patient is in the holdout')
if sel['image_id'].duplicated().any():
    GG.append('%d duplicate image_id' % int(sel['image_id'].duplicated().sum()))
if len(sel) != TARGET_IMAGES:
    GG.append('selected %d, expected %d' % (len(sel), TARGET_IMAGES))

_absent = [p for p in sel['source_path'] if not os.path.exists(p)]
print('\nselected files present on /kaggle/input : %d of %d'
      % (len(sel) - len(_absent), len(sel)))
if _absent:
    GG.append('%d selected DICOMs do not exist, e.g. %r' % (len(_absent), _absent[:2]))

print()
if GG:
    for g in GG:
        print('  GATE FAILED:', g)
    raise SystemExit('gates failed - nothing is packaged')
print('all gates passed  [PASS]')

Development patients available : 6214
Holdout patients               : 2663
Holdout overlap                : 0 [PASS]

Cancer-positive images available : 800
Cancer-negative images available : 28384

Selected malignant images : 800
Selected negative images  : 1200
Total selected images     : 2000

Unique patients           : 1423

Left/right distribution:
laterality
L    1015
R     985

CC/MLO distribution:
view
AT        1
CC      964
MLO    1035

Site distribution:
site_id
1    1169
2     831

Machine distribution:
machine_id
21     264
29     281
48     286
49     976
93      63
170     41
190      8
197      1
210     21
216     59

selected files present on /kaggle/input : 2000 of 2000

all gates passed  [PASS]


## 5. Write the metadata, then report the real size

The CSVs go out first, so the selection is reproducible even if you never package the
images. The size below is measured from the actual files, not estimated.

In [134]:
WANTED = ['patient_id', 'image_id', 'cancer', 'laterality', 'view', 'age',
          'implant', 'density', 'site_id', 'machine_id',
          'difficult_negative_case', 'dicom_path', 'partition', 'selection_type']
have = [c for c in WANTED if c in sel.columns]
gone = [c for c in WANTED if c not in sel.columns]
if gone:
    print('not in train.csv, skipped:', ', '.join(gone))

sel[have].to_csv(META_CSV, index=False)
sel[['patient_id', 'image_id', 'cancer', 'dicom_path']].to_csv(SELECT_CSV, index=False)
print('wrote', META_CSV)
print('wrote', SELECT_CSV)

sizes = [os.path.getsize(p) for p in sel['source_path']]
total = sum(sizes)
print('\nmeasured from the actual DICOMs:')
print('  total      %.2f GB' % (total / 1e9))
print('  mean       %.2f MB' % (total / len(sizes) / 1e6))
print('  min / max  %.2f / %.2f MB' % (min(sizes) / 1e6, max(sizes) / 1e6))

n_parts = int(total // int(PART_GB * 1e9)) + 1
gb = total / 1e9
print('\nwill write about %d parts of ~%.1f GB' % (n_parts, PART_GB))
print('/kaggle/working allows roughly %.0f GB; this needs %.2f GB.'
      % (WORKING_LIMIT_GB, gb))

if gb <= SAFE_GB:
    CONFIRM_PACKAGE = True
    print('  fits with %.1f GB headroom - proceeding, no confirmation needed'
          % (WORKING_LIMIT_GB - gb))
    print('\nRun section 6.')
else:
    CONFIRM_PACKAGE = False
    print('  ONLY %.1f GB of headroom. Packaging this could fail partway and leave a'
          % (WORKING_LIMIT_GB - gb))
    print('  partial set. Lower TARGET_IMAGES, or if you want to go ahead anyway set')
    print('  CONFIRM_PACKAGE = True in the next cell and re-run it.')
    print('\nStopping here rather than starting something that may not finish.')

wrote /kaggle/working/rsna_subset_2000/metadata/rsna_validation_2000_metadata.csv
wrote /kaggle/working/rsna_subset_2000/metadata/rsna_validation_2000_selection.csv

measured from the actual DICOMs:
  total      11.72 GB
  mean       5.86 MB
  min / max  0.55 / 17.05 MB

will write about 8 parts of ~1.5 GB
/kaggle/working allows roughly 20 GB; this needs 11.72 GB.
  fits with 8.3 GB headroom - proceeding, no confirmation needed

Run section 6.


## 6. Package into tar parts

Section 5 enables this automatically when the subset fits comfortably, so you normally
just run it. It only asks for confirmation when the size is near the `/kaggle/working`
cap, where packaging could fail partway.

Files are added straight from `/kaggle/input`, so there is never a second copy on disk.
A part is closed when it passes `PART_GB` and a fresh one is opened. Each finished part
is hashed. Re-running skips parts that already exist, so an interrupted session resumes.

Inside the tars the layout is `images/<patient_id>/<image_id>.dcm`, which keeps image
IDs from colliding across patients.

In [135]:
if not CONFIRM_PACKAGE:
    raise SystemExit(
        'CONFIRM_PACKAGE is False, which means section 5 found the subset too close to '
        'the /kaggle/working limit. Lower TARGET_IMAGES, or set CONFIRM_PACKAGE = True '
        'in a cell above this one and re-run if you want to proceed regardless.')

LIMIT = int(PART_GB * 1e9)
rows = sel.to_dict('records')

parts, assign = [], []
part_i, part_bytes = 0, 0
for r in rows:
    sz = os.path.getsize(r['source_path'])
    if part_bytes and part_bytes + sz > LIMIT:
        part_i += 1
        part_bytes = 0
    assign.append(part_i)
    part_bytes += sz
sel['_part'] = assign
n_parts = part_i + 1
print('packing %d images into %d parts' % (len(sel), n_parts))

t0 = time.perf_counter()
for pi in range(n_parts):
    name = os.path.join(PART_DIR, 'rsna_%d_part%02d.tar' % (TARGET_IMAGES, pi))
    group = sel[sel['_part'] == pi]
    if os.path.exists(name) and os.path.getsize(name) > 0:
        print('part %02d  exists, skipped  (%.2f GB)'
              % (pi, os.path.getsize(name) / 1e9))
        continue
    tmp = name + '.partial'
    with tarfile.open(tmp, 'w') as tf:
        for _, r in group.iterrows():
            tf.add(r['source_path'],
                   arcname='images/%s/%s.dcm' % (r['patient_id'], r['image_id']))
    os.replace(tmp, name)                       # atomic: no half-written part
    print('part %02d  %5d files  %.2f GB  %.1f min elapsed'
          % (pi, len(group), os.path.getsize(name) / 1e9,
             (time.perf_counter() - t0) / 60), flush=True)

print('\npackaged in %.1f min' % ((time.perf_counter() - t0) / 60))

packing 2000 images into 8 parts
part 00    239 files  1.49 GB  0.0 min elapsed
part 01    270 files  1.49 GB  0.2 min elapsed
part 02    243 files  1.50 GB  0.7 min elapsed
part 03    252 files  1.50 GB  1.2 min elapsed
part 04    254 files  1.50 GB  1.9 min elapsed
part 05    266 files  1.50 GB  2.6 min elapsed
part 06    261 files  1.50 GB  3.4 min elapsed
part 07    215 files  1.24 GB  4.0 min elapsed

packaged in 4.0 min


## 7. Verify and hash

Every part is reopened and its members counted, so a truncated archive is caught here
rather than on your PC. The SHA-256 of each part goes into `parts_manifest.json`.

In [136]:
def sha256_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


part_files = sorted(glob.glob(PART_DIR + '/rsna_%d_part*.tar' % TARGET_IMAGES))
manifest, counted, bad = [], 0, []
print('verifying %d parts (hashing %0.1f GB, this takes a few minutes)'
      % (len(part_files), sum(os.path.getsize(p) for p in part_files) / 1e9))
for p in part_files:
    try:
        with tarfile.open(p) as tf:
            names = [m.name for m in tf.getmembers() if m.isfile()]
    except Exception as exc:
        bad.append((os.path.basename(p), str(exc)[:100]))
        continue
    counted += len(names)
    manifest.append({'part': os.path.basename(p), 'files': len(names),
                     'bytes': os.path.getsize(p), 'sha256': sha256_file(p)})
    print('  %-28s %5d files  %.2f GB  %s'
          % (os.path.basename(p), len(names), os.path.getsize(p) / 1e9,
             manifest[-1]['sha256'][:16]))

print('\nexpected files : %d' % len(sel))
print('files in parts : %d' % counted)
print('unreadable     : %d' % len(bad))
for b in bad:
    print('   ', b)

record = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'competition': 'rsna-breast-cancer-detection',
    'environment': 'kaggle',
    'seed': SEED,
    'target_images': TARGET_IMAGES,
    'n_positive': int((sel['cancer'] == 1).sum()),
    'n_negative': int((sel['cancer'] == 0).sum()),
    'unique_patients': int(sel['patient_id'].nunique()),
    'development_patients': len(DEV),
    'holdout_patients': len(HOLD),
    'holdout_opened': False,
    'files_in_parts': counted,
    'parts': manifest,
    'metadata_csv_sha256': sha256_file(META_CSV),
    'selection_csv_sha256': sha256_file(SELECT_CSV),
}
json.dump(record, open(SUMMARY, 'w'), indent=2)
json.dump(manifest, open(MANIFEST, 'w'), indent=2)
print('\nwrote', SUMMARY)
print('wrote', MANIFEST)

if bad or counted != len(sel):
    raise SystemExit('packaging is incomplete - delete the bad part(s) and re-run '
                     'section 6; existing good parts are skipped')
print('\nPACKAGING COMPLETE [PASS]  -  %d files in %d parts'
      % (counted, len(part_files)))

verifying 8 parts (hashing 11.7 GB, this takes a few minutes)
  rsna_2000_part00.tar           239 files  1.49 GB  830d93eaf977d47e
  rsna_2000_part01.tar           270 files  1.49 GB  73ba4d00f2341dbe
  rsna_2000_part02.tar           243 files  1.50 GB  221e67facf7034c7
  rsna_2000_part03.tar           252 files  1.50 GB  846256c2fc966200
  rsna_2000_part04.tar           254 files  1.50 GB  6b45949b13584963
  rsna_2000_part05.tar           266 files  1.50 GB  d728c87b37ac7cc9
  rsna_2000_part06.tar           261 files  1.50 GB  e6c4972b1ed11f30
  rsna_2000_part07.tar           215 files  1.24 GB  aeee7d999136f72b

expected files : 2000
files in parts : 2000
unreadable     : 0

wrote /kaggle/working/rsna_subset_2000/metadata/selection_summary.json
wrote /kaggle/working/rsna_subset_2000/metadata/parts_manifest.json

PACKAGING COMPLETE [PASS]  -  2000 files in 8 parts


## 8. Getting it onto your PC

**Save Version → Save & Run All (Commit).** `/kaggle/working` is wiped when the session
ends, so an uncommitted run leaves you nothing.

Then open the finished version's **Output** tab and download each `.tar` part
individually, plus the three files in `metadata/`. Downloading parts one at a time is
the point of splitting them: a failed part is retried on its own instead of restarting
11 GB.

Avoid "Download All" — it zips everything into one ~11 GB file, which is exactly the
fragile case the parts exist to prevent.

### On your PC

Put the parts and `parts_manifest.json` in one folder and run the cell below's script.
It checks every SHA-256 before extracting, so a part that arrived corrupt is caught
before it becomes a silently incomplete dataset.

In [137]:
unpack = r"""#!/usr/bin/env python3
# Run on your PC, in the folder holding the .tar parts and parts_manifest.json.
import hashlib, json, os, sys, tarfile

man = json.load(open('parts_manifest.json'))
dest = 'rsna_subset'
os.makedirs(dest, exist_ok=True)

def sha256(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

bad, total = [], 0
for entry in man:
    p = entry['part']
    if not os.path.exists(p):
        bad.append((p, 'missing')); continue
    if os.path.getsize(p) != entry['bytes']:
        bad.append((p, 'size %d, expected %d'
                    % (os.path.getsize(p), entry['bytes']))); continue
    got = sha256(p)
    if got != entry['sha256']:
        bad.append((p, 'sha256 mismatch - re-download this part')); continue
    with tarfile.open(p) as tf:
        tf.extractall(dest)
    total += entry['files']
    print('%-28s OK  %d files' % (p, entry['files']))

print()
if bad:
    for p, why in bad:
        print('FAILED %-28s %s' % (p, why))
    sys.exit('re-download the parts above and run this again')
print('extracted %d files into %s/  [PASS]' % (total, dest))
"""

path = OUT + '/unpack_on_pc.py'
with open(path, 'w') as fh:
    fh.write(unpack)
print('wrote', path)
print()
print('Download it with the parts, then on your PC:')
print('    python unpack_on_pc.py')
print()
print('It refuses to extract a part whose hash does not match, so a corrupt download')
print('cannot quietly become a short dataset.')

wrote /kaggle/working/rsna_subset_2000/unpack_on_pc.py

Download it with the parts, then on your PC:
    python unpack_on_pc.py

It refuses to extract a part whose hash does not match, so a corrupt download
cannot quietly become a short dataset.


## 9. What this did and did not do

- selected 2,000 DICOMs from the **development** partition with `SEED = 42`
- every cancer-positive image, negatives sampled to reach the total
- copied the DICOMs into tar parts **byte-for-byte** — nothing decoded, windowed,
  resized, converted or rewritten
- the sealed holdout was never read, and the gates would have stopped the run if a
  holdout patient had reached the selection
- no model, no inference, no metrics, no train/test split

`selection_summary.json` records the counts, the seed and the SHA-256 of both CSVs, so
the same 2,000 images can be identified again without this notebook.